Import Libraries

In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA

#Plot aesthetics
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#F5F9FF",
    "axes.grid": True,
    "grid.color": "#D0E4F7",
    "grid.linewidth": 0.6,
    "font.family": "DejaVu Sans"
})
PALETTE = ["#1A3A5C", "#2196F3", "#4CAF50", "#FF9800", "#E91E63", "#9C27B0"]

print("✅  Libraries loaded successfully")

✅  Libraries loaded successfully


Load & Explore the Dataset

In [ ]:
#Load Konga dataset
df = pd.read_csv("dataset/konga_transactions.csv", parse_dates=["order_date"])

#Basic information: shape
print(f"Shape: {df.shape}")
print(f"Unique orders: {df['order_id'].nunique():,}")
print(f"Unique customers: {df['customer_id'].nunique():,}")
print(f"Date range:       {df['order_date'].min().date()}  →  {df['order_date'].max().date()}")
df.head(8)

In [ ]:
# data health check
print("--- Data Types ---")
print(df.dtypes)
print("\n--- Missing Values ---")
print(df.isnull().sum())
print("\n--- Numeric Summary ---")
df[["quantity", "unit_price_ngn", "line_total_ngn"]].describe()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Konga transactions Dataset — Exploratory Distributions", fontsize=14, fontweight="bold", y=1.02)

# Orders by city
city_counts = df.groupby("city")["order_id"].nunique().sort_values(ascending=True)
city_counts.plot(kind="barh", ax=axes[0], color=PALETTE[0])
axes[0].set_title("Orders by City"); axes[0].set_xlabel("Unique Orders")

# Revenue by category
cat_rev = df.groupby("category")["line_total_ngn"].sum().sort_values() / 1e6
cat_rev.plot(kind="barh", ax=axes[1], color=PALETTE[1])
axes[1].set_title("Revenue by Category (₦M)"); axes[1].set_xlabel("₦ Million")

# Payment method share
pay_counts = df["payment_method"].value_counts()
axes[2].pie(pay_counts, labels=pay_counts.index, autopct="%1.0f%%",
            colors=PALETTE, startangle=90, pctdistance=0.8)
axes[2].set_title("Payment Method Share")

plt.tight_layout()
plt.show()

In [ ]:

# Step 1 — Order-level aggregation
order_agg = (
    df.groupby(["customer_id", "order_id"])
      .agg(
          order_value   = ("line_total_ngn", "sum"),
          items_in_cart = ("quantity", "sum"),
          order_date    = ("order_date", "first"),
      )
      .reset_index()
)

# Step 2 — Customer-level aggregation
customer_features = (
    order_agg.groupby("customer_id")
             .agg(
                 total_orders      = ("order_id",    "nunique"),
                 total_spend_ngn   = ("order_value", "sum"),
                 avg_order_value   = ("order_value", "mean"),
                 total_items       = ("items_in_cart","sum"),
                 first_order       = ("order_date",  "min"),
                 last_order        = ("order_date",  "max"),
             )
             .reset_index()
)

# Step 3 — Category diversity per customer
cat_diversity = (
    df.groupby("customer_id")
      .agg(
          unique_categories = ("category", "nunique"),
          avg_unit_price    = ("unit_price_ngn", "mean"),
      )
      .reset_index()
)

# Step 4 — Purchase frequency (orders per month active)
customer_features["months_active"] = (
    (customer_features["last_order"] - customer_features["first_order"])
    .dt.days / 30
).clip(lower=1)
customer_features["orders_per_month"] = (
    customer_features["total_orders"] / customer_features["months_active"]
)

# Step 5 — Merge everything
customers = customer_features.merge(cat_diversity, on="customer_id")
customers = customers.drop(columns=["first_order", "last_order", "months_active"])

print(f"Customer feature matrix: {customers.shape}")
customers.head()

In [ ]:
# Feature distributions
features = ["total_orders", "total_spend_ngn", "avg_order_value",
            "total_items", "unique_categories", "avg_unit_price", "orders_per_month"]

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
fig.suptitle("Customer Feature Distributions (before scaling)", fontsize=14, fontweight="bold")
axes = axes.flatten()

for i, feat in enumerate(features):
    axes[i].hist(customers[feat], bins=30, color=PALETTE[i % len(PALETTE)], edgecolor="white")
    axes[i].set_title(feat, fontsize=10)
    axes[i].set_xlabel("Value")
    axes[i].set_ylabel("Count")

axes[-1].set_visible(False)
plt.tight_layout()
plt.show()


Preprocessing with StandardScaler 

In [ ]:
X_raw = customers[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
X_scaled_df = pd.DataFrame(X_scaled, columns=features)

# Show the effect of scaling
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Before
axes[0].boxplot([X_raw[f] for f in features], labels=features, vert=False)
axes[0].set_title("BEFORE Scaling — Raw Feature Ranges", fontweight="bold")
axes[0].set_xlabel("Raw Value")

# After
axes[1].boxplot([X_scaled_df[f] for f in features], labels=features, vert=False)
axes[1].set_title("AFTER Scaling — Normalised (mean=0, std=1)", fontweight="bold")
axes[1].set_xlabel("Scaled Value")
axes[1].axvline(0, color="red", linestyle="--", linewidth=1, alpha=0.6)

plt.tight_layout()
plt.show()

print(f"✅  Feature matrix ready: {X_scaled.shape[0]} customers × {X_scaled.shape[1]} features")

Plot the elbow curve and silhouette scores

In [31]:
K_range = range(2, 8)
inertias        = []
silhouette_scores = []

print("Computing K-Means for K = 2 to 8...")
for k in K_range:
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, labels)
    silhouette_scores.append(sil)
    print(f"  K={k}  |  Inertia={km.inertia_:>10,.1f}  |  Silhouette={sil:.4f}")

print("\n✅  Done!")

Computing K-Means for K = 2 to 8...
  K=2  |  Inertia=   6,886.0  |  Silhouette=0.4493
  K=3  |  Inertia=   4,791.1  |  Silhouette=0.4971
  K=4  |  Inertia=   3,927.1  |  Silhouette=0.5210
  K=5  |  Inertia=   3,354.1  |  Silhouette=0.4547
  K=6  |  Inertia=   2,941.8  |  Silhouette=0.4596
  K=7  |  Inertia=   2,550.5  |  Silhouette=0.3668

✅  Done!


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Choosing the Optimal K", fontsize=14, fontweight="bold")

k_list = list(K_range)

# ── Elbow plot ──
ax1.plot(k_list, inertias, "o-", color=PALETTE[0], linewidth=2.5, markersize=8)
ax1.fill_between(k_list, inertias, alpha=0.1, color=PALETTE[0])
ax1.set_title("Elbow Method — Inertia (WCSS) vs K", fontweight="bold")
ax1.set_xlabel("Number of Clusters (K)")
ax1.set_ylabel("Inertia (WCSS)")

# Mark elbow (biggest second-derivative drop)
diffs2 = np.diff(np.diff(inertias))
elbow_k = k_list[np.argmax(diffs2) + 1]
ax1.axvline(elbow_k, color="red", linestyle="--", linewidth=1.8, label=f"Elbow at K={elbow_k}")
ax1.annotate(f"  Elbow\n  K={elbow_k}", xy=(elbow_k, inertias[elbow_k-2]),
             fontsize=11, color="red", fontweight="bold")
ax1.legend()

# ── Silhouette plot ──
best_sil_k = k_list[np.argmax(silhouette_scores)]
bar_colors = [PALETTE[0] if k != best_sil_k else "#E91E63" for k in k_list]
bars = ax2.bar(k_list, silhouette_scores, color=bar_colors, edgecolor="white", width=0.6)
ax2.set_title("Silhouette Score vs K  (higher = better)", fontweight="bold")
ax2.set_xlabel("Number of Clusters (K)")
ax2.set_ylabel("Avg Silhouette Score")
ax2.set_ylim(0, max(silhouette_scores) * 1.2)

for bar, score in zip(bars, silhouette_scores):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f"{score:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

ax2.axhline(silhouette_scores[best_sil_k - 2], color="#E91E63",
            linestyle="--", linewidth=1.5, label=f"Best K={best_sil_k}")
ax2.legend()

plt.tight_layout()
plt.show()

print(f"\n📌  Elbow suggests K={elbow_k}")
print(f"📌  Silhouette score peaks at K={best_sil_k}")
print(f"\n👉  We'll proceed with K={best_sil_k} (confirmed by both methods).")
OPTIMAL_K = best_sil_k

Apply a second clustering algorithm: GMM

In [ ]:
optimal_n = 3   # adjust based on BIC/AIC plot
gmm = GaussianMixture(n_components=optimal_n, random_state=42, n_init=10)
gmm_labels = gmm.fit_predict(X_scaled_df)
gmm_probs = gmm.predict_proba(X_scaled_df)   # soft assignments

original_features = customers[features].copy()
original_features['gmm_cluster'] = gmm_labels

# Compute mean feature values per GMM cluster
gmm_profiles = original_features.groupby('gmm_cluster').mean().reset_index()
print("GMM Cluster Profiles (mean values in original units):\n")
print(gmm_profiles.round(2))

# Scaled means for heatmap
scaled_profiles_gmm = pd.DataFrame(X_raw).copy()
scaled_profiles_gmm['gmm_cluster'] = gmm_labels
scaled_means_gmm = scaled_profiles_gmm.groupby('gmm_cluster').mean().reset_index(drop=True)

# Heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(scaled_means_gmm, annot=True, cmap='RdBu_r', center=0,
            xticklabels=features, yticklabels=[f'GMM Cluster {i}' for i in range(scaled_means_gmm.shape[0])])
plt.title('GMM Cluster Profiles – Mean Feature Values (Scaled)')
plt.tight_layout()
plt.show()

1

In [ ]:
original_features = X_raw  # cols_to_scale = list of original feature names

# Add cluster labels
original_features['cluster'] = labels

# Compute mean feature values per cluster
cluster_profiles = original_features.groupby('cluster').mean().reset_index()

print("Cluster Profiles (mean values in original units):\n")
print(cluster_profiles.round(2))

# For a heatmap, we'll use the scaled means to show relative importance
# Compute scaled means per cluster (using the scaled X)
scaled_profiles = pd.DataFrame(X_scaled_df).copy()
scaled_profiles['cluster'] = labels
scaled_means = scaled_profiles.groupby('cluster').mean().reset_index(drop=True)

# Plot heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(scaled_means, annot=True, cmap='RdBu_r', center=0,
            xticklabels=features, yticklabels=[f'Cluster {i}' for i in range(scaled_means.shape[0])])
plt.title('Cluster Profiles – Mean Feature Values (Scaled)')
plt.tight_layout()
plt.show()

Compare results GMN against K-Means

In [ ]:
# Choose the number of clusters (e.g., from elbow/silhouette on K-Means)
k = 4
# 1. Fit K-Means
kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
kmeans_labels = kmeans.fit_predict(X_scaled_df)

# 2. Fit GMM (with same number of components)
gmm = GaussianMixture(n_components=k, random_state=42, covariance_type='full')
gmm_labels = gmm.fit_predict(X_scaled_df)   # hard assignment via argmax

# 3. PCA for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled_df)

# 4. Plot side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# K-Means
scatter1 = ax1.scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_labels, cmap='viridis', s=10, alpha=0.7)
ax1.set_title(f'K-Means (K={k})')
ax1.set_xlabel('PC1')
ax1.set_ylabel('PC2')
plt.colorbar(scatter1, ax=ax1, label='Cluster')

# GMM
scatter2 = ax2.scatter(X_pca[:, 0], X_pca[:, 1], c=gmm_labels, cmap='viridis', s=10, alpha=0.7)
ax2.set_title(f'GMM (K={k})')
ax2.set_xlabel('PC1')
ax2.set_ylabel('PC2')
plt.colorbar(scatter2, ax=ax2, label='Cluster')

plt.tight_layout()
plt.show()

# 5. Silhouette scores
sil_kmeans = silhouette_score(X_scaled_df, kmeans_labels)
sil_gmm = silhouette_score(X_scaled_df, gmm_labels)
print(f"Silhouette Score (K-Means): {sil_kmeans:.3f}")
print(f"Silhouette Score (GMM): {sil_gmm:.3f}")

Apply PCA to your customer feature matrix

In [ ]:
# Fit PCA without reducing dimensions
pca = PCA()
pca.fit(X_scaled_df)

# Explained variance ratio per component
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

# Number of components for 85% variance
n_components_85 = np.argmax(cumulative_variance >= 0.85) + 1

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Individual explained variance
ax1.bar(range(1, len(explained_variance) + 1), explained_variance, alpha=0.7, label='Individual')
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Explained Variance Ratio')
ax1.set_title('Explained Variance per Component')
ax1.set_xticks(range(1, len(explained_variance) + 1))

# Cumulative explained variance
ax2.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='--')
ax2.axhline(y=0.85, color='r', linestyle='-', label='85% threshold')
ax2.axvline(x=n_components_85, color='g', linestyle='--', label=f'{n_components_85} components')
ax2.set_xlabel('Number of Components')
ax2.set_ylabel('Cumulative Explained Variance')
ax2.set_title('Cumulative Explained Variance')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f"Explained variance per component: {explained_variance}")
print(f"Cumulative variance: {cumulative_variance}")
print(f"Number of components needed to explain ≥85% variance: {n_components_85}")


Re-run your best clustering algorithm in PCA-reduced space

In [46]:
pca = PCA(n_components=n_components_85, random_state=42)
X_pca = pca.fit_transform(X_scaled_df)

# 2. Fit the chosen clustering algorithm on PCA-reduced data
k = 4   # optimal number of clusters from earlier
gmm_pca = GaussianMixture(n_components=k, random_state=42, covariance_type='full')
labels_pca = gmm_pca.fit_predict(X_pca)

# 3. Compute silhouette score on the original features using these new labels
sil_pca = silhouette_score(X_scaled_df, labels_pca)
print(f"Silhouette score (original features, clusters from PCA-reduced): {sil_pca:.3f}")

# Compare with previous silhouette on original features using original-space clustering
# (We'll assume we saved that earlier as sil_gmm)
print(f"Previous silhouette (original features, original-space GMM): {sil_gmm:.3f}")


Silhouette score (original features, clusters from PCA-reduced): 0.367
Previous silhouette (original features, original-space GMM): 0.303
